In [19]:
# ===============================
# CELL 1: Prepare Data Generators
# ===============================

import os
import zipfile
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import subprocess # Import subprocess for handling .rar files

data_dir = "/content/PotatoDis"
zip_path = "/content/PotatoDis.rar" # Changed to .rar

if not os.path.exists(data_dir):
    if os.path.exists(zip_path):
        # Check if unrar is installed, install if not
        try:
            subprocess.run(["unrar", "--version"], check=True, capture_output=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            print("unrar not found, installing...")
            subprocess.run(["apt", "update"], check=True)
            subprocess.run(["apt", "install", "unrar"], check=True)
            print("unrar installed.")

        # Extract .rar file using unrar command
        try:
            subprocess.run(["unrar", "x", zip_path, "/content/"], check=True)
            print(f"Extracted {zip_path} to /content/")
        except subprocess.CalledProcessError as e:
            print(f"Error extracting {zip_path}: {e}")
    else:
        print(f"Error: RAR file not found at {zip_path}")
else:
    print(f"Directory {data_dir} already exists. Skipping extraction.")

img_height, img_width = 128, 128
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Check class indices
print("Class indices:", train_generator.class_indices)

# Optional: count images per class
unique, counts = np.unique(train_generator.classes, return_counts=True)
print("Class counts (by index):", dict(zip(unique, counts)))

Directory /content/PotatoDis already exists. Skipping extraction.
Found 1722 images belonging to 3 classes.
Found 430 images belonging to 3 classes.
Class indices: {'Potato___Early_blight': 0, 'Potato___Late_blight': 1, 'Potato___healthy': 2}
Class counts (by index): {np.int32(0): np.int64(800), np.int32(1): np.int64(800), np.int32(2): np.int64(122)}


In [20]:
# ===============================
# CELL 2: Define CNN Model
# ===============================

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Input(shape=(128, 128, 3)),
    Conv2D(32, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,923 (12.61 MB)

 Trainable params: 3,305,475 (12.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [21]:
# ===============================
# CELL 3: Train the Model (better)
# ===============================

from sklearn.utils import class_weight
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

# Compute class weights based on training data
train_labels = train_generator.classes
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_dict = dict(enumerate(class_weights))
print("Class Weights:", class_weights_dict)

# Early stopping: allow more epochs but stop when no improvement
es = EarlyStopping(
    monitor='val_loss',
    patience=10,              # more patience so model can improve
    restore_best_weights=True,
    verbose=1
)

epochs = 40  # allow more; ES will stop earlier if needed

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=epochs,
    class_weight=class_weights_dict,
    callbacks=[es]
)


Class Weights: {0: np.float64(0.7175), 1: np.float64(0.7175), 2: np.float64(4.704918032786885)}


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 21s 290ms/step - accuracy: 0.6998 - loss: 4.7169 - val_accuracy: 0.4651 - val_loss: 21.5550
Epoch 2/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 10s 180ms/step - accuracy: 0.8977 - loss: 1.1003 - val_accuracy: 0.4651 - val_loss: 42.3434
Epoch 3/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 198ms/step - accuracy: 0.8502 - loss: 1.0485 - val_accuracy: 0.4651 - val_loss: 62.5617
Epoch 4/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 201ms/step - accuracy: 0.8955 - loss: 0.4826 - val_accuracy: 0.4651 - val_loss: 76.3247
Epoch 5/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 205ms/step - accuracy: 0.9068 - loss: 0.3249 - val_accuracy: 0.4651 - val_loss: 68.8594
Epoch 6/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 202ms/step - accuracy: 0.9283 - loss: 0.2170 - val_accuracy: 0.4651 - val_loss: 57.4247
Epoch 7/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 11s 206ms/step - accuracy: 0.9155 - loss: 0.2879 - val_accuracy: 0.4651 - val_loss: 30.8523
Epoch 8/40
54/54 ━━━━━━━━━━━━━━━━━━━━ 13s 247ms/step - accuracy: 0.8954 - loss: 0.3252 - v

In [22]:
# ===============================
# CELL X: Single image prediction (no upload here)
# ===============================

import numpy as np
from tensorflow.keras.preprocessing import image

# Map index → class name using generator
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
print("Index to class:", idx_to_class)

def predict_potato_image(img_path):
    img = image.load_img(img_path, target_size=(128, 128))
    x = image.img_to_array(img)
    x = x / 255.0
    x = np.expand_dims(x, axis=0)

    preds = model.predict(x)
    pred_idx = np.argmax(preds[0])
    pred_class = idx_to_class[pred_idx]
    pred_conf = preds[0][pred_idx]

    print(f"Predicted class: {pred_class} (confidence: {pred_conf:.3f})")
    return pred_class, float(pred_conf)

# Example (only if you still want CLI testing, otherwise ignore):
# predict_potato_image("/content/some_image.jpg")


Index to class: {0: 'Potato___Early_blight', 1: 'Potato___Late_blight', 2: 'Potato___healthy'}


In [23]:
!pip install gradio -q


In [24]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing import image

idx_to_class = {v: k for k, v in train_generator.class_indices.items()}

def predict_gradio(img):
    # img is a PIL image from Gradio
    img = img.resize((128, 128))
    x = image.img_to_array(img)
    x = x / 255.0
    x = np.expand_dims(x, axis=0)

    preds = model.predict(x)
    pred_idx = int(np.argmax(preds[0]))
    pred_class = idx_to_class[pred_idx]
    pred_conf = float(preds[0][pred_idx])
    return {pred_class: pred_conf}

demo = gr.Interface(
    fn=predict_gradio,
    inputs=gr.Image(type="pil", label="Upload potato leaf"),
    outputs=gr.Label(num_top_classes=3, label="Prediction")
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://773b974f295b8f3ee8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
